In [ ]:
# --- Deep learning ---
from torch.utils.data import DataLoader
import torch

# --- Data processing ---
import os
import numpy as np
import pandas as pd
from pathlib import Path

# --- Image I/O ---
from PIL import ImageFile, Image
from skimage import io, img_as_float32

# --- Spatial transcriptomics ---
import scanpy as sc
import anndata as ad

## 1. her2st Leave-One-Out Validation (commented template)

Fixed patch half-size r=56 → 112×112 patches.

In [ ]:
Image.MAX_IMAGE_PIXELS = 100000000
exp_dir = r'./对比方法/THItoGene-main/data/her2st/data/ST-cnts'
img_dir = r'./对比方法/THItoGene-main/data/her2st/data/ST-imgs'
pos_dir = r'./对比方法/THItoGene-main/data/her2st/data/ST-spotfiles'
patch_dir = r'./对比方法/THItoGene-main/data/her2st/data/ST-patches'
r = 56

names = [file for file in os.listdir(exp_dir) if file.endswith('.tsv')]
names.sort()
names = [i[:2] for i in names]

def get_img(img_dir, name):
    pre = img_dir + '/' + name[0] + '/' + name
    fig_name = [file for file in os.listdir(pre) if file.endswith('.jpg')][0]
    path = pre + '/' + fig_name
    return Image.open(path)

def get_meta(exp_dir, pos_dir, name):
    exp_path = exp_dir + '/' + name + '.tsv'
    adata = sc.read(exp_path, delimiter='\t')
    pos_path = pos_dir + '/' + name + '_selection.tsv'
    df = pd.read_csv(pos_path, sep='\t')
    x = np.around(df['x'].values).astype(int)
    y = np.around(df['y'].values).astype(int)
    id = [str(x[i]) + 'x' + str(y[i]) for i in range(len(x))]
    df.index = id
    adata.obs = adata.obs.merge(df, how='left', left_index=True, right_index=True)
    return adata

img_dict = {i: torch.Tensor(np.array(get_img(img_dir, i))) for i in names}
exp_dict = {i: get_meta(exp_dir, pos_dir, i) for i in names}

for name in names:
    im = img_dict[name].permute(1, 0, 2)          # [H,W,C] → [W,H,C]
    centers = (np.floor(exp_dict[name].obs[['pixel_x', 'pixel_y']]).values).astype(int)
    n_patches = len(centers)
    patches = torch.zeros((n_patches, 3, r * 2, r * 2))
    for j in range(n_patches):
        center = centers[j]
        x, y = center
        patch = im[(x - r):(x + r), (y - r):(y + r), :]
        patches[j] = patch.permute(2, 0, 1)        # [W,H,C] → [C,W,H]
    print(f"Processed {name}: {patches.shape}")
    torch.save(patches, f'{patch_dir}/{name}.pt')